Bias refers to the error introduced by approximating a complex real-world problem with a simplified model, while variance refers to the model's sensitivity to fluctuations in the training data. A linear regression model has high bias and low variance; it makes strong assumptions about the data (linearity) but is stable across different datasets. If these strong assumptions are not correct, there will be places where it systematically overestimates or underestimates. In contrast, a decision tree model has low bias and high variance;it can capture complex patterns but is prone to overfitting, especially if deep and unpruned. This means that it can start to memorize the training data rather than capturing patterns that generalize.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.preprocessing import PolynomialFeatures
import numpy as np
from sklearn.model_selection import GridSearchCV

# load data
kc_sales = pd.read_csv("data/kc_house_data.csv")

Fit a linear regression model to the housing data, using sqft_living to predict price. Check the mean squared error on the training data and the test data.

In [2]:
kc_sales.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [3]:
predicator_variable = ['sqft_living']
target = 'price'

X = kc_sales[predicator_variable]
y = kc_sales[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=321, test_size=0.3
)

# Model
lr = LinearRegression().fit(X_train, y_train)

# Print results
print(f"MSE on training data: {mean_squared_error(y_train, lr.predict(X_train)):.2f}")
print(f"MSE on test data: {mean_squared_error(y_test, lr.predict(X_test)):.2f}")

MSE on training data: 69332030732.39
MSE on test data: 66079560485.17


Q2.  Repeat this but with a [DecisionTreeRegresor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html). Again check the mean squared error on the training data and the test data. How does what you see differ from the linear regression model?

In [4]:
from sklearn.model_selection import train_test_split

dt = DecisionTreeRegressor().fit(X_train, y_train)

print(f"MSE on training data: {mean_squared_error(y_train, dt.predict(X_train)):.2f}")
print(f"MSE on test data: {mean_squared_error(y_test, dt.predict(X_test)):.2f}")

MSE on training data: 50192955708.73
MSE on test data: 73538459276.97


The decision tree has a lower MSE on the training data than the linear regression model, but the test MSE is higher. This suggests that the decision tree is overfitting because it is learning the training data too closely and does not generalize as well to the test data.

Q3. Fit a DecisionTreeRegressor where you restrict the max_depth to 5. Again check the mean squared error on the training data and the test data. What do you notice now?

In [5]:
dt_depth = DecisionTreeRegressor(max_depth=5).fit(X_train, y_train)

print(f"MSE on training data: {mean_squared_error(y_train, dt_depth.predict(X_train)):.2f}")
print(f"MSE on test data: {mean_squared_error(y_test, dt_depth.predict(X_test)):.2f}")

MSE on training data: 60841449026.34
MSE on test data: 62926561451.65


After restricting the decision tree to a maximum depth of 5, the training MSE increased compared to the unrestricted tree, but the test MSE decreased and became very close to the training MSE. This indicates that the model is no longer overfitting and generalizes better to new data. Limiting the depth reduces variance while slightly increasing bias, demonstrating the bias-variance tradeoff.

When working with machine learning models, we often have to balance bias and variance. This is called the [bias-variance tradeoff](https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff). One method of this is through [regularization](https://www.ibm.com/think/topics/regularization), where we restrict the complexity of the model, adding some bias but reducing the variance, which can lead to a lower mean squared error on the test set.

Lasso and ridge regression do this by adding a penalty term based on the size of the coefficients. Smaller coefficients means that the model has less flexibility. The neat thing about these types of models is that they determine how to allocate the coefficients automatically as part of the model fitting process, so we can start with a large set of potential predictors and allow the model fitting to determine which ones to focus on.

For the next part of the exercise, we'll see how we can add complexity to our model but control the complexity through regularization.

4. So far, we've only been predicting based off of the square footage of living space. Fit a new linear regression model using all variables besides id, date, price, and zipcode. How well does this model perform on the test set compared to the model with just square footage of living space?

In [6]:
predictor_variable = kc_sales.drop(columns=[col for col in kc_sales.columns if col in ['id', 'date', 'price', 'zipcode']])
target = kc_sales['price']

X_train_all, X_test_all, y_train, y_test = train_test_split(
    predictor_variable, target, test_size=0.3, random_state=321
)

lr_all = LinearRegression().fit(X_train_all, y_train)

y_pred = lr_all.predict(X_test_all)

print(f"Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Test R²: {r2_score(y_test, y_pred):.4f}")

Test MSE: 41757655724.75
Test MAE: 125820.16
Test R²: 0.6844


Q5. Try fitting a lasso and ridge model. Becuase lasso and ridge have penalty terms based on the size of the coefficients, and the size of the coefficients depends on the scale of the variable, you'll want to scale the features first so that they are on comparable scales. Create a Pipeline object where the first step is applying a StandardScaler and the second step is either a lasso or ridge model. Because these models have a hyperparameter controlling regularization strength, you'll want to use the LassoCV and RidgeCV models, which will select values for the regularization strength using cross-validation.


In [7]:
lasso_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('lasso', LassoCV())
    ]
).fit(X_train_all, y_train)

y_pred = lasso_pipe.predict(X_test_all)

print(f"Lasso Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Lasso Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Lasso Test R²: {r2_score(y_test, y_pred):.4f}")

Lasso Test MSE: 41770331754.19
Lasso Test MAE: 125737.41
Lasso Test R²: 0.6843


In [8]:
ridge_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('ridge', RidgeCV())
    ]
).fit(X_train_all, y_train)

y_pred = ridge_pipe.predict(X_test_all)

print(f"Ridge Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Ridge Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Ridge Test R²: {r2_score(y_test, y_pred):.4f}")

Ridge Test MSE: 41757913805.39
Ridge Test MAE: 125799.41
Ridge Test R²: 0.6844


You likely didn't see much difference between the regular linear regression model and the lasso or ridge model. Let's see what happens if we add more complexity through feature interactions. We can capture more complex relationships between the predictor variables and the target variable by multiplying the predictors together before fitting the model. For example, the interaction between sqft_living and bedrooms will let the model capture if the impact of square footage depends on the number of bedrooms.


Q6. Add PolynomialFeatures to your pipeline after the standard scaler. Try using degree 2 features. How does this change the performance of the regular linear regression model, the lasso model, and the ridge model?

In [9]:
poly_lr_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('reg', LinearRegression())
    ]
).fit(X_train_all, y_train)

y_pred = poly_lr_pipe.predict(X_test_all)

print(f"Polynomial Linear Regression Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Polynomial Linear Regression Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Polynomial Linear Regression Test R²: {r2_score(y_test, y_pred):.4f}")

Polynomial Linear Regression Test MSE: 25229878727.58
Polynomial Linear Regression Test MAE: 102612.33
Polynomial Linear Regression Test R²: 0.8093


In [10]:
poly_lasso_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('lasso', LassoCV())
    ]
).fit(X_train_all, y_train)

y_pred = poly_lasso_pipe.predict(X_test_all)

print(f"Polynomial Lasso Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Polynomial Lasso Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Polynomial Lasso Test R²: {r2_score(y_test, y_pred):.4f}")

Polynomial Lasso Test MSE: 25135804989.08
Polynomial Lasso Test MAE: 102329.79
Polynomial Lasso Test R²: 0.8100


In [11]:
poly_ridge_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('ridge', RidgeCV())
    ]
).fit(X_train_all, y_train)

y_pred = poly_ridge_pipe.predict(X_test_all)

print(f"Polynomial Ridge Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Polynomial Ridge Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Polynomial Ridge Test R²: {r2_score(y_test, y_pred):.4f}")

Polynomial Ridge Test MSE: 25216536814.96
Polynomial Ridge Test MAE: 102587.78
Polynomial Ridge Test R²: 0.8094


After adding degree 2 polynomial features, the models can capture interactions between predictors. This adds complexity to the model. The regular linear regression model may improve, but it may also become more sensitive to overfitting. Lasso and Ridge help control this added complexity through regularization, so their performance may be more stable.

The lasso penalty tends to cause some coeffients to zero out, so it can be viewed as a method of automatic feature selection.

Q7. Look at the set of coefficients for the lasso model. What percentage of the coefficients are zero? What are the largest non-zero coefficients?

In [12]:
poly_lasso_pipe['lasso'].coef_

array([ 0.00000000e+00, -0.00000000e+00,  1.31720760e+04,  9.50779504e+04,
        0.00000000e+00, -3.78724641e+03,  0.00000000e+00,  0.00000000e+00,
        2.50416285e+04,  9.69536317e+04,  8.57801316e+03,  0.00000000e+00,
       -1.85496916e+04,  0.00000000e+00,  7.03014342e+04, -2.39443494e+04,
        3.85747301e+04, -0.00000000e+00,  1.80582499e+02, -0.00000000e+00,
       -2.03969236e+02, -0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00,  5.98244604e+02, -0.00000000e+00, -0.00000000e+00,
       -6.06571436e+03, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        9.66498415e+02, -0.00000000e+00, -0.00000000e+00,  5.16176706e+03,
       -0.00000000e+00, -0.00000000e+00,  1.06459315e+04,  3.20920381e+03,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  2.29638792e+03, -0.00000000e+00,  0.00000000e+00,
       -6.27306164e+03,  

In [13]:
poly_lasso_pipe[:-1].get_feature_names_out()

array(['1', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
       'sqft_living15', 'sqft_lot15', 'bedrooms^2', 'bedrooms bathrooms',
       'bedrooms sqft_living', 'bedrooms sqft_lot', 'bedrooms floors',
       'bedrooms waterfront', 'bedrooms view', 'bedrooms condition',
       'bedrooms grade', 'bedrooms sqft_above', 'bedrooms sqft_basement',
       'bedrooms yr_built', 'bedrooms yr_renovated', 'bedrooms lat',
       'bedrooms long', 'bedrooms sqft_living15', 'bedrooms sqft_lot15',
       'bathrooms^2', 'bathrooms sqft_living', 'bathrooms sqft_lot',
       'bathrooms floors', 'bathrooms waterfront', 'bathrooms view',
       'bathrooms condition', 'bathrooms grade', 'bathrooms sqft_above',
       'bathrooms sqft_basement', 'bathrooms yr_built',
       'bathrooms yr_renovated', 'bathrooms lat', 'bathrooms long',
       'bathrooms sqft_living

In [14]:
lasso_coefficients = pd.DataFrame({'variable':poly_lasso_pipe[:-1].get_feature_names_out(), 'coefficient':poly_lasso_pipe['lasso'].coef_})
lasso_coefficients

,variable,coefficient
0,1,0.000000
1,bedrooms,-0.000000
2,bathrooms,13172.076008
3,sqft_living,95077.950449
4,sqft_lot,0.000000
...,...,...
166,long sqft_living15,-9685.614544
167,long sqft_lot15,1026.233203
168,sqft_living15^2,4312.898385
169,sqft_living15 sqft_lot15,-683.007522


In [15]:
(lasso_coefficients['coefficient']!=0).sum()

np.int64(88)

The Lasso model retained 88 non-zero coefficients and set the remaining coefficients to zero. This demonstrates that Lasso performs feature selection by removing less important features. Even with many polynomial and interaction terms, only a subset of features contributes significantly to predicting house prices.

Q8. A new hyperparameter that we have is the degree of the polynomial we're using. So that we're not overfitting to the test set, we need to use cross-validation to select this value. Set up a GridSearchCV to try out polynomial degrees from 1 to 3 and to try LinearRegression, LassoCV, and RidgeCV models. Use 'neg_mean_squared_error' as the error_score. Which combination does it find does the best?

In [ ]:
pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures()),
        ('model', LinearRegression())
    ]
)

param_grid = {
    'poly__degree': [1, 2, 3],
    'model': [LinearRegression(), LassoCV(max_iter=10000), RidgeCV()]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring='neg_mean_squared_error',
    cv=5
).fit(X_train_all, y_train)



In [22]:
print(grid.best_params_)
print(f"Best MSE: {-grid.best_score_:.2f}")

{'model': LassoCV(max_iter=10000), 'poly__degree': 2}
Best MSE: 29157768309.90
